In [0]:
import os
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import pandas as pd
import json
import requests
from dotenv import load_dotenv
import time

In [0]:
# load env
load_dotenv()
AQI_KEY = os.getenv("AQI_KEY")
OPENAQ_KEY = os.getenv("OPENAQ_KEY")
username = os.getenv('GEONAMES_USER')

In [0]:
# load cities json
cities = spark.read.table('air_quality.`01_bronze`.airquality_covid_19_cities')

# Convert to pandas
cities_pd = cities.toPandas()
# get json data
cities_json = cities_pd['data'].to_json()
cities_json = json.loads(cities_json)

# Get records
records = cities_json["0"]

places_rows = []
stations_rows = []

for item in records:
    place = item.get("Place", {})
    stations = item.get("Stations", [])
    sources = item.get("Sources", [])

    country = place.get("country")
    city = place.get("name")
    feature = place.get("feature")
    city_lat, city_lon = place.get("geo", [None, None])
    population = place.get("pop")

    source_names = [s.get("name") for s in sources]

    # ---- place-level row ----
    places_rows.append({
        "country": country,
        "city": city,
        "feature": feature,
        "city_lat": city_lat,
        "city_lon": city_lon,
        "population": population,
        "n_stations": len(stations),
        "sources": source_names
    })

    # ---- station-level rows ----
    for st in stations:
        stations_rows.append({
            "country": country,
            "city": city,
            "station_name": st.get("Name"),
            "station_lat": st.get("Latitude"),
            "station_lon": st.get("Longitude")
        })

# Create DataFrames
df_places = pd.DataFrame(places_rows)
df_stations = pd.DataFrame(stations_rows)

In [0]:
def get_nigeria_stations(token):
    # Search for stations in Nigeria
    url = f"https://api.waqi.info/search/?token={token}&keyword=Nigeria"
    response = requests.get(url)
    data = response.json()

    stations_list = []

    if data['status'] == 'ok':
        for station in data['data']:
            name = station['station']['name']

            city_raw = name.split(' ')[0]
            stations_list.append({
                'country': 'NG',
                'city': city_raw,  # <--- New Column Added Here
                'station_name': name,
                'station_lat': station['station']['geo'][0],
                'station_lon': station['station']['geo'][1],
                # 'uid': station['uid']
            })

    return pd.DataFrame(stations_list)
def get_openaq_nigeria_stations(API_KEY):
    url = "https://api.openaq.org/v3/locations"

    headers = {
        "X-API-Key": API_KEY
    }

    params = {
        "countries_id": 100,   # Nigeria
        "limit": 1000,
        "page": 1
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    data = response.json()

    stations = []

    for loc in data.get("results", []):
        coords = loc.get("coordinates") or {}

        stations.append({
            "country": "NG",
            "station_name": loc.get("name"),
            "station_lat": coords.get("latitude"),
            "station_lon": coords.get("longitude"),
        })

    return pd.DataFrame(stations)

# List of major cities
major_cities = ["Ilorin", "Abuja", "Lagos", "Calabar"]

# Function to extract city
def extract_city(station_name):
    station_name_lower = station_name.lower()
    for city in major_cities:
        if city.lower() in station_name_lower:
            return city
    # Default city
    return "Lagos"

def aggregate_places(df_combined):
    """Groups stations by city to create df_places structure."""

    # 2. Group by City
    df_places = df_combined.groupby(['country', 'city']).agg(
        city_lat=('station_lat', 'mean'),
        city_lon=('station_lon', 'mean'),
        n_stations=('station_name', 'count'),
    ).reset_index()
    return df_places
def enrich_with_geonames(df_places, username):
    """Fetches population and feature code for each city."""
    print("Enriching with GeoNames data (Population/Feature)...")

    pops = []
    feats = []

    for _, row in df_places.iterrows():
        city = row['city']
        url = "http://api.geonames.org/searchJSON"
        params = {
            "q": city,
            "country": row['country'],
            "maxRows": 1,
            "username": username
        }

        try:
            resp = requests.get(url, params=params)
            data = resp.json()

            if data.get('totalResultsCount', 0) > 0:
                res = data['geonames'][0]
                pops.append(res.get('population', 0))
                feats.append(res.get('fcode', 'PPL'))
            else:
                pops.append(0)
                feats.append('PPLA') # Default placeholder

        except Exception as e:
            print(f"GeoNames Error for {city}: {e}")
            pops.append(0)
            feats.append('PPLA')

        time.sleep(0.1) # Be polite to API

    df_places['population'] = pops
    df_places['feature'] = feats
    return df_places

# Run
df_stations_nigeria_openaq = get_openaq_nigeria_stations(OPENAQ_KEY)

# Apply function to create new 'city' column
df_stations_nigeria_openaq["city"] = df_stations_nigeria_openaq["station_name"].apply(extract_city)

# Run the function
df_stations_nigeria = get_nigeria_stations(AQI_KEY)

# Concate above
df_stations_nigeria = pd.concat([df_stations_nigeria, df_stations_nigeria_openaq], ignore_index=True)
df_stations_nigeria

In [0]:
# C. Aggregate
df_places_nigeria = aggregate_places(df_stations_nigeria)

# D. Enrich with Population/Feature
df_places_nigeria = enrich_with_geonames(df_places_nigeria, username)

# E. Final Formatting to match your sample exactly
cols_order = ['country', 'city', 'feature', 'city_lat', 'city_lon', 'population', 'n_stations']
df_places_nigeria = df_places_nigeria[cols_order]

# add sources column
df_places_nigeria['sources'] = 'OPENAQ_AQICN'

In [0]:
# concatenate with existing
df_places = pd.concat([df_places, df_places_nigeria], ignore_index=True)
df_stations = pd.concat([df_stations, df_stations_nigeria], ignore_index=True)

In [0]:
df_places['sources'] = df_places['sources'].astype(str)

df_places.info()

In [0]:
spk_places = spark.createDataFrame(df_places)
spk_stations = spark.createDataFrame(df_stations)

# Write to Delta: places
(
    spk_places.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("air_quality.`02_silver`.airquality_places")
)


# Write to delta: stations
(
    spk_stations.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("air_quality.`02_silver`.airquality_stations")
)